# SKU Segmentation using ABC / XYZ Analysis

In this notebook, we segment SKUs based on:
- Business importance (revenue contribution)
- Demand predictability (volatility)

This segmentation will later decide:
- Forecasting approach
- Safety stock levels
- Replenishment frequency

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
import sys
sys.path.append('../')

DATA_PATH_PROCESSED = "../data/processed"

In [6]:
try:
    df = pd.read_csv(f"{DATA_PATH_PROCESSED}/feature_engineered_data.csv")
    print("Feature engineered data loaded successfully")
except FileNotFoundError:
    print(f"File not found at {DATA_PATH_PROCESSED}/feature_engineered_data.csv. Please ensure the file exists and the path is correct.")   
    
df["date"] = pd.to_datetime(df["date"])

Feature engineered data loaded successfully


## Revenue Contribution per SKU (ABC Analysis)

ABC analysis helps identify which SKUs contribute most to total revenue.

In [7]:
df["revenue"] = df["units_sold"] * df["selling_price"]

sku_revenue = (
    df.groupby("sku_id")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

sku_revenue["revenue_share"] = (
    sku_revenue["revenue"] / sku_revenue["revenue"].sum()
)

sku_revenue["cumulative_revenue"] = sku_revenue["revenue_share"].cumsum()

sku_revenue.head()

,sku_id,revenue,revenue_share,cumulative_revenue
0,SKU0039,21913266.76,0.060958,0.060958
1,SKU0030,18618421.10,0.051793,0.112751
2,SKU0025,17353833.06,0.048275,0.161026
3,SKU0040,17242348.29,0.047965,0.208990
4,SKU0013,16050243.67,0.044648,0.253639


## Assigning ABC Categories

- A: Top ~80% of revenue
- B: Next ~15% of revenue
- C: Remaining ~5% of revenue

In [8]:
def assign_abc(cum_rev):
    if cum_rev <= 0.80:
        return "A"
    elif cum_rev <= 0.95:
        return "B"
    else:
        return "C"

sku_revenue["ABC_class"] = sku_revenue["cumulative_revenue"].apply(assign_abc)

sku_revenue["ABC_class"].value_counts()

ABC_class
A    23
B    10
C     7
Name: count, dtype: int64

## Demand Variability per SKU (XYZ Analysis)

XYZ analysis classifies SKUs based on demand predictability
using the coefficient of variation (CV).

In [11]:
sku_volatility = (
    df.groupby("sku_id")["demand_cv"]
    .mean()
    .reset_index()
)
sku_volatility.head()

,sku_id,demand_cv
0,SKU0001,0.185876
1,SKU0002,0.221622
2,SKU0003,0.243078
3,SKU0004,0.205781
4,SKU0005,0.175662


## Assigning XYZ Categories

- X: Low variability (stable demand)
- Y: Medium variability (seasonal)
- Z: High variability (erratic demand)

In [12]:
def assign_xyz(cv):
    if cv <= 0.5:
        return "X"
    elif cv <= 1.0:
        return "Y"
    else:
        return "Z"

sku_volatility["XYZ_class"] = sku_volatility["demand_cv"].apply(assign_xyz)

sku_volatility["XYZ_class"].value_counts()

XYZ_class
X    40
Name: count, dtype: int64

## Combining ABC and XYZ Classifications

Each SKU is now assigned a combined segment such as:
- AX (high revenue, stable)
- AZ (high revenue, volatile)
- CZ (low revenue, unpredictable)

In [13]:
sku_segment = (
    sku_revenue[["sku_id", "ABC_class"]]
    .merge(sku_volatility[["sku_id", "XYZ_class"]], on="sku_id", how="left")
)

sku_segment["SKU_segment"] = (
    sku_segment["ABC_class"] + sku_segment["XYZ_class"]
)

sku_segment.head()

,sku_id,ABC_class,XYZ_class,SKU_segment
0,SKU0039,A,X,AX
1,SKU0030,A,X,AX
2,SKU0025,A,X,AX
3,SKU0040,A,X,AX
4,SKU0013,A,X,AX


## Distribution of SKU Segments

This helps understand how inventory complexity is distributed
across the catalogue.

In [14]:
sku_segment["SKU_segment"].value_counts()

SKU_segment
AX    23
BX    10
CX     7
Name: count, dtype: int64

## Business Interpretation of SKU Segments

- **AX:** Core SKUs — high priority, tight control, frequent replenishment
- **AY:** Important but seasonal — buffer stock during peaks
- **AZ:** High value but risky — higher safety stock, careful monitoring
- **BX / BY:** Medium importance — balanced control
- **CZ:** Long tail — minimal stocking, cautious replenishment

## Attaching SKU Segments to Transactional Data

The SKU segment will be used in forecasting and inventory optimization.

In [15]:
df_segmented = df.merge(
    sku_segment[["sku_id", "SKU_segment"]],
    on="sku_id",
    how="left"
)

df_segmented.head()

,date,sku_id,units_sold,selling_price,gross_revenue,promo_flag,discount_pct,month,week_of_year,day_of_week,is_weekend,is_festival_month,is_payday_period,season_tag,holiday_flag,weather_index,campaign_intensity,platform_traffic_source,traffic_index,competitor_price_index,competitor_stockout_flag,bundle_offer_flag,stock_visibility_score,rating_score,review_volume,product_visibility_rank,sku_name,category,sub_category,mrp,cost_price,supplier_id,supplier_name,avg_lead_time,lead_time_variability,week,quarter,year,is_month_start,lag_1_week_sales,lag_2_week_sales,lag_4_week_sales,rolling_4w_mean,rolling_4w_std,price_gap,is_promo,is_winter_season,is_summer_season,lead_time_days,lead_time_risk,demand_cv,revenue,SKU_segment
0,2024-01-05,SKU0001,37,812.43,30059.91,1,0.215801,1,1,4,0,0,1,winter,0,1.093677,1,organic,1.047220,0.956571,0,0,0.414885,4.99,220,20,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,True,29.0,27.0,28.0,30.0,4.760952,223.57,True,True,False,10,medium,0.158698,30059.91,AX
1,2024-01-06,SKU0001,47,645.74,30349.78,1,0.376699,1,1,5,1,0,1,winter,0,0.894081,1,paid,1.073690,1.008373,0,0,0.473478,4.99,220,6,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,False,37.0,29.0,27.0,35.0,9.092121,390.26,True,True,False,10,medium,0.259775,30349.78,AX
2,2024-01-07,SKU0001,41,553.64,22699.24,1,0.465598,1,1,6,1,0,1,winter,0,1.060748,1,organic,1.009331,0.991535,0,1,0.364909,4.99,220,88,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,1,1,2024,False,47.0,37.0,27.0,38.5,7.549834,482.36,True,True,False,10,medium,0.196100,22699.24,AX
3,2024-01-08,SKU0001,37,864.54,31987.98,1,0.165502,1,2,0,0,0,0,winter,0,0.942876,2,affiliate,0.990832,1.037452,0,0,0.388911,4.99,220,37,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,2,1,2024,False,41.0,47.0,29.0,40.5,4.725816,171.46,True,True,False,10,medium,0.116687,31987.98,AX
4,2024-01-09,SKU0001,41,882.67,36189.47,1,0.148002,1,2,1,0,0,0,winter,0,1.150076,2,affiliate,1.046387,0.998371,1,0,0.409813,4.99,220,14,NovaCharge Cable,Electronics Accessories,Audio,1036,731.48,SUP003,Skyline Imports Pvt. Ltd.,10,2,2,1,2024,False,37.0,41.0,37.0,41.5,4.123106,153.33,True,True,False,10,medium,0.099352,36189.47,AX


## Save Segmented Dataset

In [18]:
df_segmented.to_csv(
    f"{DATA_PATH_PROCESSED}/feature_engineered_with_segments.csv",
    index=False
)
print("Segmented data saved successfully")

Segmented data saved successfully
